# Sistema di Visione 3D

# 03 Calibrazione della Lama di Luce

In [1]:
import os
import cv2 as cv
import numpy as np
import pickle
import pandas as pd
import plotly.graph_objs as go
import plotly.offline as pyo
from os import listdir, path
from matplotlib import pyplot as plt
from sklearn.impute import SimpleImputer
from sklearn.cluster import DBSCAN

# Directory e costanti
IMGDIR      = 'images'
LAMA        = os.path.join(IMGDIR,'lama/cropped')
OUTDIR      = 'outputs'
Z_BASE      = 50 # altezza della base in mm

In [2]:
def vardump(var, fname):
    """
    Serializza un oggetto Python e lo salva in un file binario.

    Parametri:
      var   : Oggetto Python da salvare su disco.
      fname : Percorso e nome del file binario in cui salvare 'var'.

    Note:
      - Utilizza la libreria 'pickle' per la serializzazione.
      - La modalità 'wb' indica scrittura in binario.
    """
    with open(fname, 'wb') as fw:
        pickle.dump(var, fw)

In [ ]:
with open(os.path.join(OUTDIR, 'cameraMatrix.dump'), 'rb') as f:
    cameraMatrix = pickle.load(f)
    print("Matrice della telecamera caricata:")
    print(cameraMatrix)

with open(os.path.join(OUTDIR, 'distCoeffs.dump'), 'rb') as f:
    distCoeffs = pickle.load(f)
    print("Coefficienti di distorsione caricati:")
    print(distCoeffs)

with open(os.path.join(OUTDIR, 'newCameraMatrix.dump'), 'rb') as f:
    newcameramtx = pickle.load(f)
    print("Nuova matrice della telecamera caricata:")
    print(newcameramtx)

with open(os.path.join(OUTDIR, 'roi.dump'), 'rb') as f:
    roi = pickle.load(f)
    print("ROI caricata:")
    print(roi)

In [4]:
def cluster_laser_points(points, eps=5.0, min_samples=2000):
    """
    Esegue la clusterizzazione DBSCAN sui punti laser
    per eliminare il rumore e isolare i cluster reali corrispondenti alla lama.
    
    Parametri:
    - points: array NumPy di forma (N, 2) o (N, 3)
    - eps: distanza massima tra due punti per essere considerati nel medesimo cluster
    - min_samples: numero minimo di campioni richiesti per formare un cluster
    
    Ritorna:
    - labels: array con le etichette di cluster assegnate a ogni punto
              (-1 indica i punti considerati rumore)
    - clusters: lista di array, ciascuno contenente i punti di un cluster
    """
    db = DBSCAN(eps=eps, min_samples=min_samples).fit(points)
    labels = db.labels_
    unique_labels = set(labels)
    
    # Raggruppa i punti in base alle etichette (tranne -1 che è rumore)
    clusters = []
    for c in unique_labels:
        if c != -1:
            cluster_points = points[labels == c]
            clusters.append(cluster_points)

    # Visualizza la distribuzione dei cluster (solo per 2D)
    plt.figure(figsize=(8, 6))
    plt.clf()
    for idx, cluster_points in enumerate(clusters):
        plt.scatter(cluster_points[:, 0], cluster_points[:, 1], label=f"Cluster {idx}")

    # Evidenzia i punti di rumore (label = -1)
    noise_points = points[labels == -1]
    if len(noise_points) > 0:
        plt.scatter(noise_points[:, 0], noise_points[:, 1], c='k', marker='x', label='Noise')

    plt.title("DBSCAN Clustering della lama di luce")
    plt.legend()
    plt.show()
    
    return labels, clusters


In [5]:
def GetRedLine( imgName, z ):
    # Carica l'immagine
    img = cv.imread(imgName, cv.IMREAD_COLOR )
    assert img is not None, "ERROR: File NOT FOUND!"

    # Converti l'immagine in scala di grigi
    gray = cv.cvtColor( img, cv.COLOR_BGR2GRAY ) 
    
    # Estrai il canale rosso
    imgR = img[:, :, 0]

    # Rimuovi la distorsione dall'immagine
    undistorted = cv.undistort(gray, cameraMatrix, distCoeffs, None, newcameramtx)
    undistortedR = cv.undistort(imgR, cameraMatrix, distCoeffs, None, newcameramtx)

    # Ritaglia l'immagine
    x, y, w, h = roi
    cropped = undistorted[y:y+h, x:x+w]
    croppedR = undistortedR[y:y+h, x:x+w]

    gray = cropped.copy()
    h, w = gray.shape

    # Applica una soglia per ottenere una maschera binaria
    # ret, th2 = cv.threshold( croppedR, 240, 255, cv.THRESH_BINARY+cv.THRESH_OTSU )
    ret, th2 = cv.threshold( croppedR[:,:], 240, 255, cv.THRESH_BINARY )
    # ret, th2 = cv.adaptiveThreshold(croppedR, 255, cv.ADAPTIVE_THRESH_MEAN_C, cv.THRESH_BINARY, 11, 2)

    # Visualizza l'immagine binaria
    fig, ax = plt.subplots()
    ax.imshow( th2, cmap='gray')
    plt.tight_layout()
    plt.show()
    
    # Inizializza un array per memorizzare le coordinate della linea rossa
    linegray = np.zeros((h, 3), dtype=float)
    for j in range(h):
        nzero = np.where(th2[j, :] != 0)
        if len(nzero[0]) > 0:
            avg = np.sum(gray[j, nzero] * nzero) / np.sum(gray[j, nzero]) # Calcola la media pesata (baricentro)
            linegray[j] = np.array([avg, j, z])
        else:
            linegray[j] = np.array([np.nan, j, z])  # Gestisci i valori NaN

    # Remove or impute NaN values
    imputer = SimpleImputer(strategy='mean')
    lines = imputer.fit_transform(linegray)

    # Clusterizza i punti laser per isolare la lama
    labels, clusters = cluster_laser_points(lines, eps=3.0, min_samples=5)
        
    return linegray #lines quando applicherò il filtro

Elaborazione di tutte le immagini per ottenere un unico vettore 

In [ ]:
# Inizializza un array vuoto per memorizzare le linee rosse
lines = np.zeros((0, 3), dtype=float)

# Scansione di tutte le immagini nella directory LAMA
for f in sorted(os.listdir(LAMA)):
    if f.endswith('.jpg'):
        print(f"Elaborazione file: {f}")
        fName = os.path.join(LAMA, f)
        
        if os.path.isfile(fName):
            # Calcola la coordinata z basata sul nome del file
            z = int(f[:2]) * 10 + Z_BASE
            print(f"Calcolo della linea rossa per z = {z}")
            
            # Ottieni la linea rossa dall'immagine
            redline = GetRedLine(fName, z)
            
            # Aggiungi la linea rossa all'array lines
            lines = np.vstack([lines, redline])

In [ ]:
display( pd.DataFrame( lines ) ) # i, j, zsg

Divido i punti in due sotto-insieme: calibrazione e validazione.

In [ ]:
# Ottieni il numero di righe e colonne dell'array lines
n, _ = lines.shape

# Mescola casualmente le righe dell'array lines
np.random.shuffle(lines)

# Calcola l'indice per dividere i dati in set di calibrazione e validazione
i1 = int(n * 0.9)

# Dividi i dati in set di calibrazione (90%) e validazione (10%)
cal, val = lines[:i1, :], lines[i1:, :]

# Visualizza le forme dei set di calibrazione e validazione
print(f"Forma del set di calibrazione: {cal.shape}")
print(f"Forma del set di validazione: {val.shape}")

In [ ]:
# Carica i coefficienti della matrice m dal file 'm.dump'
m_file_path = os.path.join(OUTDIR, 'm.dump')
with open(m_file_path, 'rb') as f:
    m = pickle.load(f)

# Visualizza i coefficienti della matrice m
print("Coefficienti della matrice m caricati:")
print(m)

# Visualizza le linee rosse ottenute
print("Linee rosse ottenute:")
print(lines)

Calibrazione della Luce di Luce II

In [ ]:
# Inizializza un array vuoto per memorizzare i punti calcolati
pts = np.zeros((0, 3), dtype=float)

# Itera su tutti i punti del set di calibrazione
for pt in cal:
    # Calcola i coefficienti della matrice A
    a11 = m[0] - m[8] * pt[0]
    a12 = m[1] - m[9] * pt[0]
    a21 = m[4] - m[8] * pt[1]
    a22 = m[5] - m[9] * pt[1]

    # Costruisci la matrice A
    A = np.array([[a11, a12], [a21, a22]])

    # Calcola i termini noti del sistema lineare
    b1 = pt[0] - (m[2] - m[10] * pt[0]) * pt[2] - m[3]
    b2 = pt[1] - (m[6] - m[10] * pt[1]) * pt[2] - m[7]
    b = np.array([[b1], [b2]])

    # Risolvi il sistema lineare A * x = b
    x = np.linalg.solve(A, b)

    # Aggiungi il punto calcolato all'array pts
    pts = np.vstack([pts, (x[0, 0], x[1, 0], pt[2])])

# Visualizza i punti calcolati (xsg, ysg, zsg di ogni punto)
print("Punti calcolati (xsg, ysg, zsg):")
print(pts)

In [ ]:
# Inizializza un array vuoto per memorizzare i punti di validazione calcolati
vals = np.zeros((0, 3), dtype=float)

# Itera su tutti i punti del set di validazione
for pt in val:
    # Calcola i coefficienti della matrice A
    a11 = m[0] - m[8] * pt[0]
    a12 = m[1] - m[9] * pt[0]
    a21 = m[4] - m[8] * pt[1]
    a22 = m[5] - m[9] * pt[1]

    # Costruisci la matrice A
    A = np.array([[a11, a12], [a21, a22]])

    # Calcola i termini noti del sistema lineare
    b1 = pt[0] - (m[2] - m[10] * pt[0]) * pt[2] - m[3]
    b2 = pt[1] - (m[6] - m[10] * pt[1]) * pt[2] - m[7]
    b = np.array([[b1], [b2]])

    # Risolvi il sistema lineare A * x = b
    x = np.linalg.solve(A, b)

    # Aggiungi il punto calcolato all'array vals
    vals = np.vstack([vals, (x[0, 0], x[1, 0], pt[2])])

# Visualizza i punti di validazione calcolati (xsg, ysg, zsg di ogni punto)
print("Punti di validazione calcolati (xsg, ysg, zsg):")
print(vals)

In [ ]:
# Inizializza la modalità notebook per Plotly
pyo.init_notebook_mode()

# Crea il grafico 3D per i punti di calibrazione
tCal = go.Scatter3d(
    x=pts[:, 0], 
    y=pts[:, 1], 
    z=pts[:, 2], 
    mode='markers', 
    marker={'size': 2, 'opacity': 0.8},
    name='Calibration Points'
)

# Crea il grafico 3D per i punti di validazione
tVal = go.Scatter3d(
    x=vals[:, 0], 
    y=vals[:, 1], 
    z=vals[:, 2], 
    mode='markers', 
    marker={'size': 2, 'opacity': 0.8},
    name='Validation Points'
)

# Definisci il layout del grafico
layout = go.Layout(
    margin={'l': 0, 'r': 0, 'b': 0, 't': 0},
    scene=dict(
        xaxis=dict(title='X (mm)'),
        yaxis=dict(title='Y (mm)'),
        zaxis=dict(title='Z (mm)')
    )
)

# Crea la figura con i dati e il layout
data = [tCal, tVal]
fig = go.Figure(data=data, layout=layout)

# Visualizza il grafico
pyo.iplot(fig)

In [ ]:
# Visualizza le prime due colonne dei punti calcolati (xsg, ysg)
print("Prime due colonne dei punti calcolati (xsg, ysg):")
print(pts[:, 0:2])

# Visualizza la terza colonna dei punti calcolati (zsg)
print("Terza colonna dei punti calcolati (zsg):")
print(pts[:, 2])

# Crea una colonna di 1 per l'intercetta
ones = np.ones((pts.shape[0], 1))

# Costruisci la matrice A concatenando le prime due colonne dei punti calcolati e la colonna di 1
A = np.hstack([pts[:, 0:2], ones])

# Costruisci il vettore b con la terza colonna dei punti calcolati
b = pts[:, 2]

# Visualizza la matrice A
print("Matrice A (xsg, ysg, 1):")
print(A)

# Visualizza il vettore b (zsg)
print("Vettore b (zsg):")
print(b)

In [ ]:
# Risolvi il sistema lineare A * x = b utilizzando il metodo dei minimi quadrati
x, res, rank, sing = np.linalg.lstsq(A, b, rcond=None)

# Visualizza i coefficienti A, B, C ottenuti dalla soluzione del sistema lineare
print("Coefficienti A, B, C ottenuti dalla soluzione del sistema lineare:")
print(x)

# salvataggio dei coefficienti A,B,C del piano
vardump(x, path.join(OUTDIR, 'lightPlane.dump'))